# Sigmoid Neuron Explorer

This notebook trains a sigmoid neuron on one of five datasets and lets you explore the results through interactive graphs.

### How to use
1. **Run Cell 1** to load all the functions (you will not see any output — that is expected).
2. **Run Cell 2** to choose a dataset and train the neuron. Training may take a moment.
3. **Run Cell 3** to explore the graphs once training is complete.

---

In [ ]:
import csv
import random
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def load_csv(path):
    with open(path, "r", newline="") as file:
        file_reader = csv.reader(file)
        header = next(file_reader)
        rows = []
        for r in file_reader:
            if not r:
                continue
            row = []
            for value in r:
                number_value = float(value)
                row.append(number_value)
            rows.append(row)
    feature_data = []
    for r in rows:
        feature_data.append(r[:-1])
    label_data = []
    for r in rows:
        value = int(r[-1])
        label_data.append(value)
    feature_name = header[:-1]
    label_name = header[-1]
    return feature_data, label_data, feature_name, label_name

def get_data_list(data, indexes):
    lst = []
    for i in indexes:
        lst.append(data[i])
    return lst

def train_test_split(features, labels, test_ratio, seed_value):
    index = []
    length = len(features)
    for i in range(length):
        index.append(i)
    seed = random.Random(seed_value)
    seed.shuffle(index)
    cut = int(length * (1 - test_ratio))
    train_index = index[:cut]
    test_index = index[cut:]
    features_train = get_data_list(features, train_index)
    labels_train = get_data_list(labels, train_index)
    features_test = get_data_list(features, test_index)
    labels_test = get_data_list(labels, test_index)
    return features_train, labels_train, features_test, labels_test

def activation_function(z):
    prediction = 1 / (1 + np.exp(-z))
    return prediction

def dot_product_one_vector(features_row, weights, bias):
    total = 0
    for i in range(len(features_row)):
        total += weights[i] * features_row[i]
    total += bias
    return total

def predict_one_vector(features_row, weights, bias):
    total = dot_product_one_vector(features_row, weights, bias)
    return activation_function(total)

def set_up_weights(num_features):
    weights = []
    for value in range(num_features):
        weights.append(0.0)
    return weights

def train(path, learning_rate, epochs):
    x, y, feature_names, label_name = load_csv(path)
    x_train, y_train, x_test, y_test = train_test_split(x, y, test_ratio=0.3, seed_value=75)

    num_features = len(x_train[0])
    weights = set_up_weights(num_features)
    bias = 0.0
    weight_history = []
    loss_history = []
    max_weight_change_history = []

    for epoch in range(epochs):
        weightChangeList = []
        lossList = []
        for i in range(len(x_train)):
            weightChangeList.append([])
            features_row = x_train[i]
            true_label = y_train[i]
            pred_label = predict_one_vector(features_row, weights, bias)
            for j in range(num_features):
                change_in_weights = learning_rate * (pred_label - true_label) * features_row[j]
                weightChangeList[i].append(abs(change_in_weights))
                weights[j] = weights[j] - change_in_weights
            change_in_bias = learning_rate * (pred_label - true_label)
            weightChangeList[i].append(abs(change_in_bias))
            bias -= change_in_bias
            weight_history.append((weights.copy(), bias))
        for k in range(len(x_train)):
            features_row = x_train[k]
            true_label = y_train[k]
            pred_label = predict_one_vector(features_row, weights, bias)
            if true_label == 1:
                error = -1 * np.log(pred_label)
                lossList.append(error)
            elif true_label == 0:
                error = -1 * np.log(1 - pred_label)
                lossList.append(error)
        epoch_bce = sum(lossList) / len(lossList)
        loss_history.append(epoch_bce)
        all_changes = []
        for change_row in weightChangeList:
            for change_value in change_row:
                all_changes.append(change_value)
        max_weight_change_history.append(max(all_changes))

    return x_train, y_train, weight_history, loss_history, max_weight_change_history, num_features

def show_decision_boundary(x_train, y_train, weight_history, frame):
    features = np.array(x_train)
    labels = np.array(y_train)
    weights, bias = weight_history[frame]
    w1, w2 = weights

    is_Zero = [v == 0 for v in labels]
    is_One = [v == 1 for v in labels]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(features[is_Zero, 0], features[is_Zero, 1], c='red', label='Class 0', edgecolors='k')
    ax.scatter(features[is_One, 0], features[is_One, 1], c='green', label='Class 1', edgecolors='k')

    ax.set_xlim(features[:, 0].min() - 1, features[:, 0].max() + 1)
    ax.set_ylim(features[:, 1].min() - 1, features[:, 1].max() + 1)

    if w2 != 0:
        x_vals = np.array(ax.get_xlim())
        y_vals = -(w1 * x_vals + bias) / w2
        ax.plot(x_vals, y_vals, 'k--', lw=2, label='Decision Boundary')
    elif w1 != 0:
        x_boundary = -bias / w1
        ax.axvline(x=x_boundary, color='k', linestyle='--', lw=2, label='Decision Boundary')

    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(f'Decision Boundary — Update {frame + 1} of {len(weight_history)}')
    ax.legend()
    plt.tight_layout()
    plt.show()

def show_loss_graph(loss_history):
    epochs = range(1, len(loss_history) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, loss_history, label='BCE Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.set_title('Loss Over Epochs')
    ax.legend()
    plt.tight_layout()
    plt.show()

def show_weight_change_graph(max_weight_change_history):
    epochs = range(1, len(max_weight_change_history) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, max_weight_change_history, label='Max Weight Change')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Max Absolute Weight Change')
    ax.set_title('Weight Change Over Epochs')
    ax.legend()
    plt.tight_layout()
    plt.show()

print("Functions loaded successfully. Proceed to Cell 2.")

Functions loaded successfully. Proceed to Cell 2.


---
## Step 2 — Select a Dataset and Train

Use the dropdown to choose a dataset, then click **Train** to run the sigmoid neuron. Training will print progress as it runs.

In [ ]:
dataset_options = {
    "Dataset 1": "dataset1.csv",
    "Dataset 2": "dataset2.csv",
    "Dataset 3": "dataset3.csv",
    "Dataset 4": "dataset4.csv",
    "Dataset 5": "dataset5.csv",
}

learning_rate = 0.01
epochs = 1000

dataset_dropdown = widgets.Dropdown(
    options=list(dataset_options.keys()),
    description="Dataset:",
    style={"description_width": "initial"}
)

train_button = widgets.Button(
    description="Train",
    button_style="primary"
)

train_output = widgets.Output()

training_results = {}

def on_train_clicked(b):
    with train_output:
        clear_output()
        selected = dataset_dropdown.value
        path = dataset_options[selected]
        print(f"Training on {selected} ({path})...")
        x_train, y_train, weight_history, loss_history, max_weight_change_history, num_features = train(path, learning_rate, epochs)
        training_results["x_train"] = x_train
        training_results["y_train"] = y_train
        training_results["weight_history"] = weight_history
        training_results["loss_history"] = loss_history
        training_results["max_weight_change_history"] = max_weight_change_history
        training_results["num_features"] = num_features
        print(f"Training complete! Final loss: {loss_history[-1]:.4f}")
        print("Proceed to Cell 3 to explore the graphs.")

train_button.on_click(on_train_clicked)
display(dataset_dropdown, train_button, train_output)

Dropdown(description='Dataset:', options=('Dataset 1', 'Dataset 2', 'Dataset 3', 'Dataset 4', 'Dataset 5'), st…

Button(button_style='primary', description='Train', style=ButtonStyle())

Output()

---
## Step 3 — Explore the Graphs

Use the buttons below to view different graphs. For the Decision Boundary, use the slider to scrub through each weight update and watch the boundary move.

> **Make sure you have trained a dataset in Cell 2 before running this cell.**

In [3]:
btn_boundary = widgets.Button(description="Decision Boundary", button_style="info", layout=widgets.Layout(width="200px"))
btn_loss = widgets.Button(description="Loss Over Epochs", button_style="info", layout=widgets.Layout(width="200px"))
btn_weights = widgets.Button(description="Weight Change", button_style="info", layout=widgets.Layout(width="200px"))

graph_output = widgets.Output()

def on_boundary_clicked(b):
    with graph_output:
        clear_output(wait=True)
        if not training_results:
            print("Please train a dataset first in Cell 2.")
            return
        if training_results["num_features"] != 2:
            print("Decision boundary is only available for datasets with 2 features.")
            return
        total_frames = len(training_results["weight_history"])
        slider = widgets.IntSlider(
            value=0,
            min=0,
            max=total_frames - 1,
            step=1,
            description="Update:",
            continuous_update=False,
            layout=widgets.Layout(width="600px"),
            style={"description_width": "initial"}
        )
        slider_output = widgets.Output()

        def on_slider_change(change):
            with slider_output:
                clear_output(wait=True)
                show_decision_boundary(
                    training_results["x_train"],
                    training_results["y_train"],
                    training_results["weight_history"],
                    change["new"]
                )

        slider.observe(on_slider_change, names="value")
        display(slider, slider_output)
        show_decision_boundary(
            training_results["x_train"],
            training_results["y_train"],
            training_results["weight_history"],
            0
        )

def on_loss_clicked(b):
    with graph_output:
        clear_output(wait=True)
        if not training_results:
            print("Please train a dataset first in Cell 2.")
            return
        show_loss_graph(training_results["loss_history"])

def on_weights_clicked(b):
    with graph_output:
        clear_output(wait=True)
        if not training_results:
            print("Please train a dataset first in Cell 2.")
            return
        show_weight_change_graph(training_results["max_weight_change_history"])

btn_boundary.on_click(on_boundary_clicked)
btn_loss.on_click(on_loss_clicked)
btn_weights.on_click(on_weights_clicked)

display(widgets.HBox([btn_boundary, btn_loss, btn_weights]), graph_output)

Output()